
# Test / Inference Notebook – Satellite Imagery Based Property Valuation

This notebook:
- Loads the trained multimodal model
- Runs inference on the test dataset
- Generates final predictions CSV (id, predicted_price)


In [ ]:

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.config import cfg
from src.datasets import HouseDataset
from src.model import FusionModel
from src.data_fetcher import download


In [ ]:

# Load test data
test_df = pd.read_excel(cfg.test_xlsx)
test_df.head()


In [ ]:

# Download / load satellite images for test data
img_paths = download(test_df)


In [ ]:

# Load trained model
ckpt = torch.load(
    f"{cfg.model_dir}/best_model.pt",
    map_location=cfg.device
)

model = FusionModel(tab_in=len(cfg.tab_feats)).to(cfg.device)
model.load_state_dict(ckpt['model'])
model.eval()


In [ ]:

# Prepare test dataset
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Restore scaler statistics from checkpoint
scaler.mean_ = ckpt['scaler_mean']
scaler.scale_ = ckpt['scaler_scale']

test_ds = HouseDataset(test_df, img_paths, scaler, train=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)


In [ ]:

# Run inference
preds, ids = [], []

with torch.no_grad():
    for img, tab, pid in test_loader:
        img = img.to(cfg.device)
        tab = tab.to(cfg.device)

        pred = model(img, tab).cpu().numpy()
        pred = np.expm1(pred)  # reverse log transform

        preds.extend(pred.squeeze().tolist())
        ids.extend(pid.tolist())


In [ ]:

# Save predictions
submission = pd.DataFrame({
    "id": ids,
    "predicted_price": preds
})

submission.to_csv(
    f"{cfg.output_dir}/submission.csv",
    index=False
)

submission.head()



## Output
- `outputs/submission.csv`  
Contains final predictions in the required format:
```
id, predicted_price
```
